> ### 💾 Saving Your Work — Read This First
>
> If you close this tab without saving, your work will be lost.
>
> **How to save:** **File → Download → Download .ipynb** and keep the file somewhere safe.
> Re-upload it to Colab if you need to come back to it later.
>
> **Tip:** Save at the end of every section — about every 20 minutes.


# Homework 6: State Radio Deep Dive — A Briefing for Chairman Carr

**CSC 2053 — Fall 2026 · Due Monday, November 17, 2026 at 11:59 PM ET**

---

*You've just been assigned to the research staff of **FCC Chairman Brendan Carr**.
He's preparing for an oversight hearing and needs a data-driven briefing on the
radio landscape in your assigned state — ownership concentration, signal reach,
sports coverage, and geographic gaps.*

*Your job: 5 Python warm-up questions followed by 25 Chairman Carr questions,
using data from the RadioLand app.*

**Note:** there are five written responses — to Q2, Q6, Q12, Q16 and Q25 —
that the FCC chairman would like you to fill in with your own observations.

---

In this assignment you will use **Pandas**, **NumPy**, **Matplotlib** and **Seaborn**
to analyze your state's radio stations. You'll also build an interactive map
with **Folium** and calculate station distances using the **Haversine formula**.

> **Save often.** When finished: **File → Download → Download .ipynb** and upload the file to **Brightspace** under the HW 6 submission folder.


---
## Warm-Up — Python Basics (5 Questions)

Five quick questions to get your fingers back on the keyboard before the big pandas work begins.
These test the exact primitives — variables, lists, dicts, loops — that pandas builds on top of.


---
### Warm-Up 1 — Variables
**Concept:** variable assignment, snake_case, data types

Judy is a 6-year-old Airedale Terrier who weighs 45 pounds and lives in Burlington County, NJ.

Assign each fact to a Python variable using snake_case, then print each variable on its own line.


In [ ]:
dog_name   = ''
breed      = ''
age        = 0
weight_lbs = 0.0
county     = ''

# Print each variable on its own line:


---
### Warm-Up 2 — f-strings
**Concept:** f-string formatting, the `:,` specifier

Using the variables below, write a **single** `print` statement with an f-string that produces this exact output:

```
Villanova University, PA — Go Wildcats! Founded 1842, enrollment 10,768.
```

Use `:,` inside the braces for the enrollment comma.


In [ ]:
university = 'Villanova'
state      = 'PA'
founded    = 1842
mascot     = 'Wildcats'
enrollment = 10768

# Write your single f-string print statement:


---
### Warm-Up 3 — Lists
**Concept:** list creation, indexing, negative indexing, `len()`, for loop

Create a list of at least **six** northeastern U.S. county names. Burlington County, NJ must be included.

- Print the first county
- Print the last county using a negative index
- Print the total count
- Loop through and print each county name


In [ ]:
northeast_counties = []   # add at least 6 — Burlington required

# First county:

# Last county (negative index):

# Total count:

# Loop and print each:


---
### Warm-Up 4 — Lists of Dictionaries
**Concept:** list of dicts, bracket access, loop, finding max

A pandas DataFrame is essentially a list of dictionaries under the hood — one dict per row.
This warmup uses that exact shape.

- Print the name and population of the first county
- Loop and print each county in the format: `Burlington County, NJ — pop. 461,860`
- Find and print the most populous county


In [ ]:
counties = [
    {'name': 'Burlington',  'state': 'NJ', 'population': 461860},
    {'name': 'Middlesex',   'state': 'NJ', 'population': 825061},
    {'name': 'Suffolk',     'state': 'NY', 'population': 1525920},
    {'name': 'Fairfield',   'state': 'CT', 'population': 943332},
    {'name': 'Montgomery',  'state': 'PA', 'population': 856553},
]

# a) Name and population of first county:

# b) Loop and print in target format:

# c) Find and print the most populous county:


---
### Warm-Up 5 — For Loop with If / elif / else
**Concept:** for loop, if / elif / else, arithmetic

In pandas you'll filter rows with boolean masks. This warmup does the same filtering manually with a for loop — the operation you're replacing when you reach pandas.

Judy the Airedale has had six vet checkups. Her target weight is 45 lbs.

- Loop through her weights and print each with a label:
  - `Underweight` — below 40 lbs
  - `Healthy` — 40 to 50 lbs (inclusive)
  - `Overweight` — above 50 lbs
- Format: `42.0 lbs — Healthy`
- After the loop, print her average weight using `:.1f`


In [ ]:
checkup_weights = [38.5, 42.0, 45.0, 51.5, 44.0, 45.0]

# a) Loop and label each weight:

# b) Average weight:


---
## End of Warm-Up — The Chairman Carr Briefing Begins Below

Everything above was vanilla Python. From here on you'll be using **Pandas**, **NumPy**, and the RadioLand state data to answer 25 questions.


In [ ]:
# Run this cell first — installs folium if needed, then imports everything
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "folium", "-q"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import math
from IPython.display import display

print("All libraries loaded successfully.")


---
## Setup — Loading and Preparing Your State's Data

Complete all four setup steps before answering any questions.

### Step 1 — Load Your Assigned State


In [ ]:
STATE = "XX"   # <-- Replace XX with your two-letter state abbreviation (e.g., "TX", "PA", "OH")

url = f"https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab10-python-through-state-radio/main/states/{STATE}_stations.csv"
df = pd.read_csv(url)

print(f"Loaded {len(df):,} stations for {STATE}")
df.head()


### Step 2 — Remove Non-Conforming Callsigns

U.S. broadcast callsigns begin with **K** or **W**. Rows where the callsign
starts with **D** are non-conforming entries that should be removed.


In [ ]:
bad_rows = df[df['callsign'].str.startswith('D', na=False)]
print(f"Non-conforming rows to remove: {len(bad_rows)}")

df = df[~df['callsign'].str.startswith('D', na=False)].reset_index(drop=True)
print(f"Stations after cleaning: {len(df):,}")


### Step 3 — Fix Longitude

The dataset stores longitude as a **positive** number. In the Western Hemisphere
(where the U.S. is), longitude is **negative**. Negate the `lon` column now
so that all geographic filtering works correctly.


In [ ]:
print("Before fix — sample lon values:", df['lon'].head(3).tolist())

df['lon'] = df['lon'] * -1

print("After fix  — sample lon values:", df['lon'].head(3).tolist())


### Step 4 — Create Separate AM and FM DataFrames

You'll use `am_df` and `fm_df` throughout this assignment.


In [ ]:
am_df = df[df['band'] == 'AM'].copy().reset_index(drop=True)
fm_df = df[df['band'] == 'FM'].copy().reset_index(drop=True)

print(f"AM stations : {len(am_df):,}")
print(f"FM stations : {len(fm_df):,}")
print(f"Total       : {len(df):,}")


---
## Section 1 — Initial Reconnaissance

*Chairman Carr needs a quick summary of the landscape before diving deeper.*


### Question 1 — Station Count by Band

How many total radio stations are in your state?
Display the total count and a breakdown by band (AM vs FM) using `.value_counts()`.


In [ ]:
# Q1 — YOUR CODE HERE


### Question 2 — First Look at the Data

Display the **first 5 rows** of `df`.
Then print all column names and their data types using `.dtypes`.


In [ ]:
# Q2a — First 5 rows
# YOUR CODE HERE


In [ ]:
# Q2b — Column names and data types
# YOUR CODE HERE


**Observation:** Which column do you think will be most useful for Chairman Carr's ownership investigation?
What does the data tell you about how it's structured?

*(Double-click this cell to type your answer here.)*

> **Your answer:**


### Question 3 — County Coverage

Using `df` (all stations), how many **unique county values** are represented in the dataset?
Which county has the **most stations**? Show the top 10.

*Hint: `.nunique()` counts distinct values; `.value_counts()` ranks them.*

> **Note:** Your count may be slightly higher than the number of counties in your state.
> Some radio transmitters are physically located just across a state border — the county
> column reflects where the transmitter actually sits, not where the station is licensed.

In [ ]:
# Q3 — YOUR CODE HERE


### Question 4 — Missing Slogans

Using `df` (all stations), what **percentage** are missing a `slogan`?
Round to 1 decimal place.

*Hint: `.isna().sum()` counts null values. Divide by `len(df)` and multiply by 100.*


In [ ]:
# Q4 — YOUR CODE HERE


### Question 5 — Top 5 Formats

List the **top 5 most common programming formats** across all stations (AM + FM combined),
sorted by count.


In [ ]:
# Q5 — YOUR CODE HERE


---
## Section 2 — Signal Intelligence: Power & Reach

*Chairman Carr wants to know who is dominating the airwaves — and who can barely be heard.*


### Question 6 — Power Statistics (NumPy)

Using **NumPy**, calculate the **mean, median, min, max, and standard deviation**
of the `power` column for AM stations and FM stations **separately**. Print both sets of results.

*Hint: use `.dropna().to_numpy()` to get a clean NumPy array from a Pandas column.*


In [ ]:
# Q6 — YOUR CODE HERE


**Observation:** Are AM or FM stations more powerful on average in your state?
Does this surprise you given AM's reputation for long-range coverage?

*(Double-click this cell to type your answer here.)*

> **Your answer:**


### Question 7 — High-Power FM Stations

Using **NumPy boolean indexing** on `fm_df`, how many FM stations broadcast at
**50 kW ERP or more** (the FCC's maximum for commercial FM)?
List their callsign, city, power, and format, sorted descending by power.


In [ ]:
# Q7 — YOUR CODE HERE


### Question 8 — Frequency Conversion (Vectorized NumPy)

FM frequencies are stored in MHz. Using a **NumPy vectorized operation** (no loops),
add a new column called `freq_khz` to `fm_df` that converts frequency from MHz to kHz.

Display the first 5 rows showing `callsign`, `frequency`, and `freq_khz`.


In [ ]:
# Q8 — Convert FM frequency from MHz to kHz
fm_df = fm_df.copy()   # avoids a Pandas SettingWithCopyWarning — keep this line
fm_df['freq_khz'] = # YOUR CODE HERE

fm_df[['callsign', 'frequency', 'freq_khz']].head(5)


### Question 9 — Average HAAT (FM)

**HAAT** (Height Above Average Terrain) measures how high a transmitter sits
relative to the surrounding terrain — a key factor in signal reach.

Using `fm_df`, what is the average `haat` for FM stations in your state?
Drop null values before calculating.


In [ ]:
# Q9 — YOUR CODE HERE


---
## Section 3 — Who's Running the Airwaves?

*One of Chairman Carr's top priorities is understanding media ownership concentration
in local radio markets. Let's see who controls the dial in your state.*


### Question 10 — Top 5 Owners

Who are the **top 5 owners** in your state by total station count (AM + FM combined)?
Display owner name and station count, sorted descending.


In [ ]:
# Q10 — YOUR CODE HERE


### Question 11 — Dominant Market

What is the most dominant **radio market** in your state by station count?
Show the **top 3 markets** overall, then for the #1 market show the AM vs FM breakdown.


In [ ]:
# Q11 — YOUR CODE HERE


### Question 12 — Multi-County Owners

Using `df` (AM + FM combined), are there any owners who hold licenses in
**more than 3 different counties**? List them with their county count, sorted descending.

*Hint: `.groupby('owner')['county'].nunique()` gives you county reach per owner.*


In [ ]:
# Q12 — YOUR CODE HERE


**Observation:** What does the multi-county ownership data tell you about media consolidation
in your state? If you were briefing Chairman Carr, what would you highlight?

*(Double-click this cell to type your answer here.)*

> **Your answer:**


---
## Section 4 — Game Day on the Radio: Sports Affiliates

*Chairman Carr has publicly emphasized the importance of local sports broadcasting
as a public interest issue. Let's audit your state's sports radio coverage.*


### Question 13 — NFL Affiliates

How many stations in your state carry an **NFL affiliate**?
List each unique NFL team and the stations (callsign + city) that carry it.

The starter code below handles the grouping and printing — you write the filter.


In [ ]:
# Q13 — Filter to stations with a non-null, non-empty 'nfl' column
nfl_stations = # YOUR CODE HERE

print(f"Stations with an NFL affiliate in {STATE}: {len(nfl_stations)}")
print()

if len(nfl_stations) > 0:
    for team, group in nfl_stations.groupby('nfl'):
        print(f"  {team} ({len(group)} station{'s' if len(group) > 1 else ''})")
        for _, row in group.iterrows():
            print(f"    {row['callsign']} — {row['city']}")
else:
    print("No NFL affiliates found in this state's data.")


### Question 14 — Sports Summary Table

Using `df` (all stations), build a summary table showing how many stations carry
each type of sports affiliate: NFL, MLB, NBA, NHL.
Which sport has the broadest radio presence in your state?


In [ ]:
# Q14 — YOUR CODE HERE
# Your output should display each sport and its station count, sorted descending.


### Question 15 — Multi-Sport Stations

Are there any stations that carry **more than one sports affiliation**
(any combination of NFL, MLB, NBA, NHL)?

The `.apply()` pattern below is new — read it carefully, then fill in the filter.


In [ ]:
# Q15 — Stations with 2+ sports affiliates
sports_cols = ['nfl', 'mlb', 'nba', 'nhl']

# This counts how many sports columns are non-null and non-empty for each row
df['sports_count'] = df[sports_cols].apply(
    lambda row: sum(1 for v in row if pd.notna(v) and v != ''), axis=1
)

multi_sport = # YOUR CODE HERE — filter df where sports_count >= 2

print(f"Stations with 2+ sports affiliates in {STATE}: {len(multi_sport)}")
if len(multi_sport) > 0:
    display(multi_sport[['callsign', 'city'] + sports_cols])
else:
    print("No stations found with multiple sports affiliates.")

df.drop(columns=['sports_count'], inplace=True)


### Question 16 — Most Sports-Connected City

Using `df`, which **city** in your state has the most sports-affiliated radio stations total?

*Hint: filter `df` to rows where at least one sports column is non-null and non-empty, then `.value_counts()` on `city`.*


In [ ]:
# Q16 — YOUR CODE HERE


**Observation:** Chairman Carr has argued that local sports broadcasting is a public interest issue.
Does your state's data suggest sports fans have adequate radio coverage?
Are there major teams or regions you'd flag as underserved?

*(Double-click this cell to type your answer here.)*

> **Your answer:**


---
## Section 5 — Geographic Intel: Bounding Box Analysis

*Chairman Carr wants a close look at a specific region of your state —
not just the whole picture.*

### Question 17 — Draw Your Bounding Box

Go to **[bboxfinder.com](http://bboxfinder.com)** and draw a rectangle
around a **major metro area or region** in your state.

The site gives you coordinates in this order: `lon_min, lat_min, lon_max, lat_max`.
U.S. longitude is **negative** — bboxfinder will give you the correct negative values directly.

Fill in the four values below, then write the filter.


In [ ]:
# Q17 — Define your bounding box coordinates (from bboxfinder.com)
lat_min = # YOUR VALUE  (e.g., 39.83)
lat_max = # YOUR VALUE  (e.g., 39.97)
lon_min = # YOUR VALUE  (e.g., -75.55 — must be negative!)
lon_max = # YOUR VALUE  (e.g., -75.25 — must be negative!)

# Filter df to stations within the box
in_box = df[
    (df['lat'] >= lat_min) & (df['lat'] <= lat_max) &
    (df['lon'] >= lon_min) & (df['lon'] <= lon_max)
].copy()

am_in_box = len(in_box[in_box['band'] == 'AM'])
fm_in_box = len(in_box[in_box['band'] == 'FM'])

print(f"Stations in bounding box : {len(in_box)}")
print(f"  AM : {am_in_box}")
print(f"  FM : {fm_in_box}")
print()
in_box[['callsign', 'band', 'city', 'frequency', 'format', 'lat', 'lon']].head(10)


### Question 18 — Inside the Box

Within your bounding box:
1. What are the **top 3 most common formats**?
2. Which single station has the **highest ERP power**?


In [ ]:
# Q18 — YOUR CODE HERE


### Question 19 — Folium Map of Your Bounding Box

Using **Folium**, create an interactive map of all stations within your bounding box.

- **Blue circle markers** = FM stations
- **Red circle markers** = AM stations
- Each marker popup shows: callsign, format, and power

The map structure is provided — fill in the marker color and popup text.


In [ ]:
# Q19 — Folium interactive map
center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

for _, row in in_box.iterrows():
    if pd.isna(row['lat']) or pd.isna(row['lon']):
        continue

    color = # YOUR CODE HERE — 'blue' if FM, 'red' if AM

    popup_text = # YOUR CODE HERE — a string showing callsign, format, and power

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200),
    ).add_to(m)

legend_html = (
    '<div style="position:fixed; bottom:30px; left:30px; z-index:1000; background:white;'
    ' padding:10px; border:2px solid grey; border-radius:5px; font-size:13px;">'
    '<b>Legend</b><br>'
    '<span style="color:blue;">&#9679;</span> FM Station<br>'
    '<span style="color:red;">&#9679;</span> AM Station'
    '</div>'
)
m.get_root().html.add_child(folium.Element(legend_html))

m


---
## Section 6 — Distance from the Seat of Power

*How far are your state's radio stations from the state capital?
Chairman Carr wants to know if the capital city is well-served — or if stations
cluster elsewhere.*

### The Haversine Formula

The **Haversine formula** calculates the great-circle distance between two
latitude/longitude points on the surface of a sphere. It is provided for you below.

A lookup table of state capital coordinates is also provided — your state pulls automatically.


In [ ]:
# Capital coordinates lookup — (latitude, longitude)
CAPITALS = {
    "AL": (32.361538, -86.279118), "AK": (58.301935, -134.419740),
    "AZ": (33.448457, -112.073844), "AR": (34.736009, -92.331122),
    "CA": (38.555605, -121.468926), "CO": (39.7391667, -104.984167),
    "CT": (41.767, -72.677), "DE": (39.161921, -75.526755),
    "FL": (30.4518, -84.27277), "GA": (33.76, -84.39),
    "HI": (21.30895, -157.826182), "ID": (43.613739, -116.237651),
    "IL": (39.783250, -89.650373), "IN": (39.790942, -86.147685),
    "IA": (41.590939, -93.620866), "KS": (39.04, -95.69),
    "KY": (38.197274, -84.86311), "LA": (30.45809, -91.140229),
    "ME": (44.323535, -69.765261), "MD": (38.972945, -76.501157),
    "MA": (42.2352, -71.0275), "MI": (42.7335, -84.5467),
    "MN": (44.95, -93.094), "MS": (32.32, -90.207),
    "MO": (38.572954, -92.189283), "MT": (46.595805, -112.027031),
    "NE": (40.809868, -96.675345), "NV": (39.160949, -119.753877),
    "NH": (43.220093, -71.549127), "NJ": (40.221741, -74.756138),
    "NM": (35.667231, -105.964575), "NY": (42.659829, -73.781339),
    "NC": (35.771, -78.638), "ND": (46.813343, -100.779004),
    "OH": (39.962245, -83.000647), "OK": (35.482309, -97.534994),
    "OR": (44.931109, -123.029159), "PA": (40.269789, -76.875613),
    "RI": (41.82355, -71.422132), "SC": (34.000, -81.035),
    "SD": (44.367966, -100.336378), "TN": (36.165, -86.784),
    "TX": (30.266667, -97.75), "UT": (40.7547, -111.892622),
    "VT": (44.26639, -72.57194), "VA": (37.54, -77.46),
    "WA": (47.042418, -122.893077), "WV": (38.349497, -81.633294),
    "WI": (43.074722, -89.384444), "WY": (41.145548, -104.802042),
}

if STATE not in CAPITALS:
    print(f"Capital coordinates for {STATE} not found.")
else:
    cap_lat, cap_lon = CAPITALS[STATE]
    print(f"State capital coordinates for {STATE}: ({cap_lat}, {cap_lon})")


### Question 20 — Add `distance_to_capital_miles` Column

Using the Haversine function below, add a new column `distance_to_capital_miles` to `df`.
Each value should be the distance from that station's transmitter to your state capital.
Round to 2 decimal places.


In [ ]:
# Q20 — Haversine function (provided) + add distance column
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth's radius in miles
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return R * 2 * math.asin(math.sqrt(a))

# Apply haversine to every row using df.apply()
# Each row's lat/lon is compared against cap_lat and cap_lon
df['distance_to_capital_miles'] = df.apply(
    lambda row: round(haversine(row['lat'], row['lon'], cap_lat, cap_lon), 2)
    if pd.notna(row['lat']) and pd.notna(row['lon']) else None,
    axis=1
)

print("Column added. Preview:")
df[['callsign', 'city', 'band', 'distance_to_capital_miles']].head(10)


### Question 21 — Closest and Furthest Stations

What are the **5 stations closest** to your state capital?
What are the **5 furthest**?

Display each as a table with callsign, city, band, format, and distance.


In [ ]:
# Q21 — YOUR CODE HERE


### Question 22 — Average Distance by Band

Using **NumPy**, calculate the average `distance_to_capital_miles` for AM stations
and FM stations separately.

Is FM coverage more concentrated near the capital, or does distance vary similarly for both bands?


In [ ]:
# Q22 — YOUR CODE HERE
# Hint: update am_df and fm_df first so they include the new distance column,
# then use .dropna().to_numpy() and np.mean()


---
## Section 7 — Visualizing the Airwaves

*A picture is worth a thousand rows. Let's make Chairman Carr's briefing visual.*


### Question 23 — FM Power Distribution (Matplotlib Histogram)

Using **Matplotlib**, create a histogram of FM station `power` (ERP) distribution.
- Add a vertical **dashed line** for the mean power
- Title: *"FM Station Power Distribution — [STATE]"*
- Label both axes


In [ ]:
# Q23 — Matplotlib histogram
fig, ax = plt.subplots(figsize=(9, 5))

# YOUR CODE HERE — draw the histogram and the mean line

ax.set_title(f"FM Station Power Distribution — {STATE}", fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel("Power (kW ERP)", fontsize=12)
ax.set_ylabel("Number of Stations", fontsize=12)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()


### Question 24 — Top 10 Formats (Seaborn Bar Chart)

Using **Seaborn**, create a horizontal bar chart of the **top 10 most common
programming formats** across all stations (AM + FM combined).

- Title: *"Top 10 Formats in [STATE]"*
- Label both axes


In [ ]:
# Q24 — Seaborn horizontal bar chart
fig, ax = plt.subplots(figsize=(9, 6))

# YOUR CODE HERE — build top_formats DataFrame and call sns.barplot()

ax.set_title(f"Top 10 Formats in {STATE}", fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel("Number of Stations", fontsize=12)
ax.set_ylabel("Format", fontsize=12)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()


### Question 25 — Geographic Dot Map (Scatter Plot)

Create a **scatter plot** of station `lat` vs `lon`, colored by band:
- **Red** = AM stations
- **Blue** = FM stations

Add a legend and a descriptive title.


In [ ]:
# Q25 — Geographic dot map
fig, ax = plt.subplots(figsize=(10, 7))

# YOUR CODE HERE — plot AM and FM as separate scatter series with different colors

ax.set_title(f"Radio Station Geographic Distribution — {STATE}", fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()


**Observation:** Where are stations clustered and where are the gaps?
If Chairman Carr asked you to recommend one underserved region for a new broadcast license,
where would you point — and why?

*(Double-click this cell to type your answer here.)*

> **Your answer:**


---
## Submission

1. Fill in your name below
2. Run every cell top-to-bottom one more time: **Runtime → Run all**
3. **File → Download → Download .ipynb**
4. Upload the downloaded `.ipynb` file to **Brightspace** under the HW 6 submission folder


In [ ]:
student_name = ""   # First Last
state_chosen = STATE

print(f"HW 6 submission — {student_name} — State: {state_chosen}")
